# ICCD753 - Recuperación de Información 2026-A
## Examen Final: Sistema RAG sobre arXiv Paper Abstracts

**Estudiante:** Bryan Yunga Jumbo
**Profesor:** Iván Carrera (EPN-FIS)

**Aplicación desplegada:** https://examen-final-debvy2oa93cdyd2mq57rym.streamlit.app

## A. Preparación del corpus

El corpus utilizado es la colección **arXiv Paper Abstracts**, obtenida desde Kaggle
(`spsayakpaul/arxiv-paper-abstracts`). El dataset incluye dos archivos; se seleccionó
`arxiv_data_210930-054931.csv` por tener mayor cobertura de documentos únicos
(41,105 documentos frente a 38,972 del archivo alternativo), verificado mediante
la intersección de títulos entre ambos archivos.

Se aplicaron las siguientes transformaciones:
1. **Eliminación de duplicados** por título, ya que un mismo artículo repetido no
   aporta información nueva y podría sesgar las métricas de recuperación.
2. **Renombrado de columnas** a nombres estándar (`title`, `abstract`, `categories`)
   para mayor claridad del código.
3. **Creación de un identificador único (`doc_id`)** por documento, necesario más
   adelante para vincular cada evidencia recuperada con su documento original.
4. No se realizó división (*chunking*) de los documentos, ya que cada abstract es
   un texto corto que no excede el límite de tokens de los modelos de embeddings
   utilizados, por lo que cada abstract completo constituye una unidad de recuperación.

In [1]:
import kagglehub

# Si el dataset es público, muchas veces funciona sin login,
# pero si pide autenticación, kagglehub te pedirá tu Kaggle username/key
# la primera vez que ejecutes esto en el runtime
path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'arxiv-paper-abstracts' dataset.
Path to dataset files: /kaggle/input/arxiv-paper-abstracts


In [2]:
import os
path = "/kaggle/input/arxiv-paper-abstracts"
print(os.listdir(path))

['arxiv_data_210930-054931.csv', 'arxiv_data.csv']


In [3]:
import pandas as pd

path = "/kaggle/input/arxiv-paper-abstracts"

df1 = pd.read_csv(f"{path}/arxiv_data.csv")
df2 = pd.read_csv(f"{path}/arxiv_data_210930-054931.csv")

print("arxiv_data.csv ->", df1.shape)
print(df1.columns.tolist())
print()
print("arxiv_data_210930-054931.csv ->", df2.shape)
print(df2.columns.tolist())

arxiv_data.csv -> (51774, 3)
['titles', 'summaries', 'terms']

arxiv_data_210930-054931.csv -> (56181, 3)
['terms', 'titles', 'abstracts']


In [4]:
# Revisamos duplicados y nulos en cada dataset
print("=== arxiv_data.csv (df1) ===")
print("Nulos:\n", df1.isnull().sum())
print("Duplicados (por título):", df1['titles'].duplicated().sum())

print("\n=== arxiv_data_210930-054931.csv (df2) ===")
print("Nulos:\n", df2.isnull().sum())
print("Duplicados (por título):", df2['titles'].duplicated().sum())

# Vemos si los títulos de df1 están todos contenidos en df2
titulos_df1 = set(df1['titles'])
titulos_df2 = set(df2['titles'])
interseccion = titulos_df1 & titulos_df2
print(f"\nTítulos en común: {len(interseccion)} de {len(titulos_df1)} en df1")

# Y un vistazo rápido a una fila de cada uno
print("\nEjemplo df2:")
print(df2.iloc[0])

=== arxiv_data.csv (df1) ===
Nulos:
 titles       0
summaries    0
terms        0
dtype: int64
Duplicados (por título): 12802

=== arxiv_data_210930-054931.csv (df2) ===
Nulos:
 terms        0
titles       0
abstracts    0
dtype: int64
Duplicados (por título): 15076

Títulos en común: 38865 de 38972 en df1

Ejemplo df2:
terms                                                ['cs.LG']
titles       Multi-Level Attention Pooling for Graph Neural...
abstracts    Graph neural networks (GNNs) have been widely ...
Name: 0, dtype: object


In [5]:
import pandas as pd

path = "/kaggle/input/arxiv-paper-abstracts"

# Cargamos únicamente el dataset elegido (más completo y con mayor cobertura)
df = pd.read_csv(f"{path}/arxiv_data_210930-054931.csv")

# Eliminamos duplicados por título (mismo paper no debe repetirse en el corpus)
df = df.drop_duplicates(subset="titles").reset_index(drop=True)

# Renombramos columnas a nombres estándar y más claros
df = df.rename(columns={
    "titles": "title",
    "abstracts": "abstract",
    "terms": "categories"
})

# Creamos un identificador único por documento (evidencia trazable)
df["doc_id"] = df.index.astype(str)

print("Forma final del corpus:", df.shape)
df.head(3)

Forma final del corpus: (41105, 4)


,categories,title,abstract,doc_id
0,['cs.LG'],Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...,0
1,"['cs.LG', 'cs.AI']",Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...,1
2,"['cs.LG', 'cs.CR', 'stat.ML']",Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...,2


In [6]:
# Longitud de cada abstract en número de palabras
df["abstract_len"] = df["abstract"].apply(lambda x: len(str(x).split()))

print(df["abstract_len"].describe())
print("\nAbstracts con menos de 15 palabras:", (df["abstract_len"] < 15).sum())
print(df[df["abstract_len"] < 15][["title", "abstract"]].head(5))

count    41105.000000
mean       171.467340
std         45.954856
min          5.000000
25%        140.000000
50%        169.000000
75%        201.000000
max        498.000000
Name: abstract_len, dtype: float64

Abstracts con menos de 15 palabras: 3
                                                   title  \
8713   Diverse Behavior Is What Game AI Needs: Genera...   
17761  Fast Efficient Object Detection Using Selectiv...   
37811  A Machine-Learning Framework for Design for Ma...   

                                                abstract  
8713                       this paper has been withdrawn  
17761            Retraction due to significant oversight  
37811  this is a duplicate submission(original is arX...  


Se identificaron 3 registros cuyo campo `abstract` no correspondía a un resumen
científico real, sino a notas administrativas (ej. "this paper has been withdrawn",
"Retraction due to significant oversight"). Estos casos se eliminaron aplicando un
filtro de longitud mínima (menos de 20 palabras), umbral conservador dado que el
percentil 25 de longitud de abstracts reales en el corpus es de 140 palabras, muy
por encima del filtro aplicado. Esto evita que notas administrativas contaminen la
búsqueda semántica sin arriesgar la pérdida de abstracts legítimos.

In [7]:
# Eliminamos registros con abstracts inválidos (notas administrativas, no contenido real)
df = df[df["abstract_len"] >= 20].reset_index(drop=True)

# Ya no necesitamos esta columna auxiliar
df = df.drop(columns=["abstract_len"])

# Reasignamos doc_id tras el filtrado, para que sea consistente y sin huecos
df["doc_id"] = df.index.astype(str)

print("Forma final del corpus tras limpieza:", df.shape)
df.head(3)

Forma final del corpus tras limpieza: (41096, 4)


,categories,title,abstract,doc_id
0,['cs.LG'],Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...,0
1,"['cs.LG', 'cs.AI']",Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...,1
2,"['cs.LG', 'cs.CR', 'stat.ML']",Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...,2


## B. Representación mediante embeddings

Se evaluaron cuatro modelos de embeddings de código abierto: `all-MiniLM-L6-v2`,
`BAAI/bge-small-en-v1.5`, `BAAI/bge-base-en-v1.5` e `intfloat/e5-base`. La comparación
consideró: dimensión del vector resultante (impacto en almacenamiento y velocidad de
búsqueda), tiempo de codificación sobre una muestra del corpus, y calidad reportada en
benchmarks públicos de recuperación (MTEB).

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import time

# Tomamos una muestra pequeña del corpus para comparar rendimiento sin esperar horas
muestra = df["abstract"].sample(500, random_state=42).tolist()

modelos_a_probar = [
    "all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
]

resultados = []

for nombre_modelo in modelos_a_probar:
    print(f"Probando: {nombre_modelo} ...")
    modelo = SentenceTransformer(nombre_modelo)

    inicio = time.time()
    embeddings = modelo.encode(muestra, show_progress_bar=False)
    duracion = time.time() - inicio

    resultados.append({
        "modelo": nombre_modelo,
        "dimension": embeddings.shape[1],
        "tiempo_500_docs_seg": round(duracion, 2)
    })

import pandas as pd
resultados_df = pd.DataFrame(resultados)
print(resultados_df)

Se seleccionó **BAAI/bge-small-en-v1.5** como modelo de embeddings. La comparación
empírica sobre una muestra de 500 documentos mostró que `bge-base` triplica el tiempo
de codificación de `bge-small` (8.51s vs 2.91.71s) sin una mejora que justifique el costo
computacional adicional para este caso de uso. Frente a `all-MiniLM-L6-v2`, `bge-small`
mostró una diferencia de velocidad mínima (2.91s vs 1.93s) mientras que reporta mejor
calidad de recuperación en benchmarks públicos (MTEB), manteniendo la misma dimensión
del vector (384), lo que favorece un índice vectorial liviano y adecuado para despliegue
en un entorno con recursos limitados.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os
os.makedirs("data", exist_ok=True)
# Cargamos el modelo elegido
modelo_embeddings = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Codificamos TODO el corpus (puede tardar varios minutos con 41,096 documentos)
# normalize_embeddings=True es importante: normaliza los vectores para que la
# similitud coseno se pueda calcular correctamente más adelante
embeddings_corpus = modelo_embeddings.encode(
    df["abstract"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64
)

print("Forma de los embeddings generados:", embeddings_corpus.shape)

# Guardamos los embeddings para no tener que regenerarlos cada vez
os.makedirs("models", exist_ok=True)
np.save("models/embeddings_corpus.npy", embeddings_corpus)

# También guardamos el dataframe limpio correspondiente, para mantenerlos sincronizados
df.to_csv("data/corpus_limpio.csv", index=False)

print("Embeddings y corpus guardados correctamente.")

In [11]:
print(os.listdir("models"))
print(os.listdir("data"))

['embeddings_corpus.npy']
['corpus_limpio.csv']


## C. Almacenamiento y búsqueda vectorial

Se comparó FAISS, ChromaDB y Pinecone considerando costo, dependencia de servicios
externos y facilidad de despliegue. Se seleccionó **FAISS** por ser una solución
completamente local (sin dependencia de credenciales ni conectividad externa durante
la inferencia), ser el estándar de facto para corpus de este tamaño (~41,000 documentos),
e integrarse de forma nativa con LangChain. Se utilizó el wrapper `FAISS` de
`langchain_community.vectorstores`, que asocia metadata (título, categorías, doc_id)
a cada vector, permitiendo recuperar el contenido completo del documento junto con
el resultado de la búsqueda.

In [ ]:
import os

!pip install -q langchain langchain-community faiss-cpu

# Ensure langchain_core is available, it's usually a dependency of langchain_community
# If not, it can be installed separately: !pip install -q langchain-core
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.embeddings.base import Embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

# LangChain espera una clase Embeddings con métodos embed_documents / embed_query.
# Creamos un wrapper simple sobre nuestro modelo de sentence-transformers,
# para poder reutilizar el mismo modelo (bge-small) dentro del ecosistema LangChain.
class BGEEmbeddings(Embeddings):
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        # BGE recomienda un prefijo de instrucción SOLO para las consultas,
        # no para los documentos, para mejorar la calidad de la búsqueda
        instruccion = "Represent this sentence for searching relevant passages: "
        return self.model.encode(instruccion + text, normalize_embeddings=True).tolist()

embedding_function = BGEEmbeddings()

# Convertimos cada fila del corpus en un objeto Document de LangChain,
# separando el contenido (abstract, lo que se busca semánticamente)
# de la metadata (lo que se muestra como evidencia)
documentos = [
    Document(
        page_content=row["abstract"],
        metadata={
            "doc_id": row["doc_id"],
            "title": row["title"],
            "categories": row["categories"]
        }
    )
    for _, row in df.iterrows()
]

# Construimos el índice FAISS reutilizando los embeddings YA calculados en el Paso 2,
# para no tener que volver a codificar 41,096 documentos (ahorra ~4 minutos)
texto_y_embedding = list(zip(df["abstract"].tolist(), embeddings_corpus.tolist()))
metadatas = [doc.metadata for doc in documentos]

vector_store = FAISS.from_embeddings(
    text_embeddings=texto_y_embedding,
    embedding=embedding_function,
    metadatas=metadatas
)

# Guardamos el índice en disco para no reconstruirlo cada vez
vector_store.save_local("models/faiss_index")

print("Índice FAISS creado y guardado con éxito.")
print("Número de vectores en el índice:", vector_store.index.ntotal)

## D. Recuperación (Búsqueda Vectorial + Re-ranking)

Para cumplir estrictamente con los requerimientos del examen y obtener una recuperación de alta precisión, se implementó un pipeline en **dos etapas**:

1. **Búsqueda Vectorial Inicial (High Recall):** Utilizando el índice FAISS y nuestro modelo Bi-Encoder (`BAAI/bge-small-en-v1.5`), recuperamos los 15 documentos más relevantes (`fetch_k = 15`). Esta etapa es ultra-rápida pero puede incluir falsos positivos debido a similitudes semánticas superficiales.
2. **Re-ranking con Cross-Encoder (High Precision):** Pasamos los 15 candidatos iniciales por el modelo `cross-encoder/ms-marco-MiniLM-L-6-v2`. A diferencia de los embeddings estáticos, el Cross-Encoder evalúa la consulta y el documento simultáneamente mediante atención cruzada, re-ordenándolos para seleccionar únicamente los `k` mejores pasajes para el contexto del LLM.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings

# 1. Cargamos la función de embeddings y el índice FAISS
class BGEEmbeddings(Embeddings):
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        instruccion = "Represent this sentence for searching relevant passages: "
        return self.model.encode(instruccion + text, normalize_embeddings=True).tolist()

embedding_function = BGEEmbeddings()
vector_store = FAISS.load_local("models/faiss_index", embedding_function, allow_dangerous_deserialization=True)
print("✅ Índice FAISS cargado con éxito.")

# 2. Cargamos el modelo de Re-ranking (Cross-Encoder)
print("⏳ Cargando modelo de Re-ranking...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Cross-Encoder cargado con éxito.")

# 3. Función de recuperación en 2 etapas
def retrieve_documents(query, k=5, fetch_k=15):
    """
    Recupera fetch_k candidatos con FAISS y los re-ordena con el Cross-Encoder
    para devolver los top k definitivos con su score de relevancia.
    """
    # Etapa 1: Bi-Encoder (FAISS)
    initial_docs_with_scores = vector_store.similarity_search_with_score(query, k=fetch_k)
    initial_docs = [doc for doc, _ in initial_docs_with_scores]

    # Etapa 2: Cross-Encoder (Re-ranking)
    pairs = [[query, doc.page_content] for doc in initial_docs]
    rerank_scores = reranker.predict(pairs)

    # Emparejar y ordenar de mayor a menor relevancia
    docs_with_rerank_scores = list(zip(initial_docs, rerank_scores))
    docs_with_rerank_scores.sort(key=lambda x: x[1], reverse=True)

    retrieved_top_k = docs_with_rerank_scores[:k]

    print(f"\n🔍 Recuperados y re-ordenados {len(retrieved_top_k)} documentos para: '{query}'")
    for i, (doc, score) in enumerate(retrieved_top_k):
        print(f"\n--- Documento {i+1} (Score Re-rank: {score:.4f}) ---")
        print(f"Título: {doc.metadata.get('title', 'N/A')}")
        print(f"Categorías: {doc.metadata.get('categories', 'N/A')}")
        print(f"Abstract (primeras 200 chars): {doc.page_content[:200]}...")

    return retrieved_top_k

# Ejemplo de prueba:
# retrieve_documents("What are the main applications of Graph Neural Networks?", k=3)

## E. Generación aumentada por recuperación (RAG)

En esta etapa, construiremos el pipeline de Generación Aumentada por Recuperación (RAG). Esto implica tomar los documentos recuperados en el paso anterior y usarlos como contexto para un Large Language Model (LLM) con el fin de generar una respuesta coherente y fundamentada. Seleccionaremos un LLM apropiado para la tarea y configuraremos un encadenamiento (chain) que combine la recuperación con la generación.

In [14]:
!pip install -q google-genai

In [15]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

assert os.environ.get("GOOGLE_API_KEY"), "❌ No se encontró la API Key. Revisa el gestor de Secrets."
print("✅ API Key de Gemini detectada correctamente.")

✅ API Key de Gemini detectada correctamente.


In [32]:
import os
from google import genai

# Configuración del cliente LLM (Gemini)
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))
LLM_MODEL = "gemini-3.5-flash"

RAG_PROMPT_TEMPLATE = """Eres un asistente experto en literatura científica. Tu tarea es responder la consulta del usuario ÚNICAMENTE utilizando la información contenida en los documentos de contexto proporcionados a continuación.

Reglas:
1. Si el contexto contiene información suficiente, responde de forma clara y completa, integrando información de varios documentos si es necesario.
2. Cuando uses información de un documento, referencia su número entre corchetes, por ejemplo [Doc 1], [Doc 2].
3. Si el contexto NO contiene información suficiente para responder la consulta, indícalo explícitamente diciendo: "El corpus no contiene información suficiente para responder esta consulta con certeza." No inventes información que no esté en el contexto.

--- CONTEXTO ---
{context}
--- FIN DEL CONTEXTO ---

Consulta del usuario: {query}

Respuesta:"""

# Nota: Reutilizamos retrieve_documents(), vector_store y reranker cargados en la Sección D.

def build_context(retrieved_documents):
    context_parts = []
    for i, doc in enumerate(retrieved_documents):
        titulo = doc.metadata.get("title", "N/A")
        categorias = doc.metadata.get("categories", "N/A")
        abstract = doc.page_content
        context_parts.append(
            f"[Doc {i+1}] Título: {titulo}\nCategorías: {categorias}\nAbstract: {abstract}"
        )
    return "\n\n".join(context_parts)


def generate_answer(query, context):
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, query=query)
    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config={
            "temperature": 0.2,
            "system_instruction": "Eres un asistente de investigación riguroso y honesto.",
        },
    )
    return response.text


def rag_query(query, k=5):
    """
    Pipeline RAG completo: Recuperación en 2 etapas + Contexto + LLM.
    """
    docs_with_scores = retrieve_documents(query, k=k, fetch_k=k*3)
    retrieved_documents = [doc for doc, _ in docs_with_scores]
    context = build_context(retrieved_documents)
    answer = generate_answer(query, context)

    evidencias = [
        {
            "titulo": doc.metadata.get("title", "N/A"),
            "categorias": doc.metadata.get("categories", "N/A"),
            "fragmento": doc.page_content[:300] + "...",
            "score": round(float(score), 4), # Score real del Cross-Encoder
        }
        for doc, score in docs_with_scores
    ]

    return {
        "query": query,
        "respuesta": answer,
        "evidencias": evidencias,
    }

# Prueba rápida del pipeline completo:
resultado = rag_query("What are the main applications of Graph Neural Networks?", k=3)
print("\nRESPUESTA GENERADA:\n", resultado["respuesta"])


🔍 Recuperados y re-ordenados 3 documentos para: 'What are the main applications of Graph Neural Networks?'

--- Documento 1 (Score Re-rank: 7.5404) ---
Título: Hierarchical Bipartite Graph Convolution Networks
Categorías: ['cs.LG', 'cs.CV', 'stat.ML']
Abstract (primeras 200 chars): Recently, graph neural networks have been adopted in a wide variety of
applications ranging from relational representations to modeling irregular data
domains such as point clouds and social graphs. H...

--- Documento 2 (Score Re-rank: 6.1202) ---
Título: Neural Bipartite Matching
Categorías: ['cs.LG', 'stat.ML']
Abstract (primeras 200 chars): Graph neural networks (GNNs) have found application for learning in the space
of algorithms. However, the algorithms chosen by existing research (sorting,
Breadth-First search, shortest path finding, ...

--- Documento 3 (Score Re-rank: 5.8683) ---
Título: Training Sensitivity in Graph Isomorphism Network
Categorías: ['cs.LG', 'cs.SI', 'stat.ML']
Abstract (primeras 2

## F. Presentación de evidencias

Cada respuesta generada por el sistema va acompañada de las evidencias que la
sustentan, para que el usuario pueda verificar la relación entre su consulta,
los documentos recuperados y la respuesta generada. Por cada documento usado
como contexto se muestra: título, categorías (temática arXiv), un fragmento
del abstract, y el **score de similitud coseno** respecto a la consulta (a
mayor score, mayor cercanía semántica).

Es importante notar que FAISS, con el índice por defecto (`IndexFlatL2`),
retorna internamente una **distancia euclidiana (L2)** y no una similitud
coseno directa. Dado que los vectores están normalizados
(`normalize_embeddings=True`), ambas magnitudes se relacionan mediante:

`distancia_L2² = 2 − 2·similitud_coseno`

Por eso, antes de mostrar el score al usuario, se convierte explícitamente
a similitud coseno (`1 - distancia² / 2`), evitando que se interprete al
revés (un score de distancia bajo significa *más* parecido, no menos).

Esto se implementa en la función `rag_query()` (sección E), que retorna un
diccionario con la respuesta y una lista estructurada de evidencias,
consumida directamente por la interfaz (sección G).


In [17]:
# Reutilizamos rag_query() de la sección E para mostrar cómo se presentan
# las evidencias de forma explícita, incluyendo el score de similitud coseno
resultado_demo = rag_query("What are the main applications of Graph Neural Networks?", k=3)

print("RESPUESTA GENERADA:\n", resultado_demo["respuesta"])
print("\n--- EVIDENCIAS UTILIZADAS ---")
for i, ev in enumerate(resultado_demo["evidencias"]):
    print(f"\n[Doc {i+1}]")
    print(f"Título: {ev['titulo']}")
    print(f"Categorías: {ev['categorias']}")
    print(f"Similitud coseno: {ev['score']}")
    print(f"Fragmento: {ev['fragmento']}")



🔍 Recuperados y re-ordenados 3 documentos para: 'What are the main applications of Graph Neural Networks?'

--- Documento 1 (Score Re-rank: 7.5404) ---
Título: Hierarchical Bipartite Graph Convolution Networks
Categorías: ['cs.LG', 'cs.CV', 'stat.ML']
Abstract (primeras 200 chars): Recently, graph neural networks have been adopted in a wide variety of
applications ranging from relational representations to modeling irregular data
domains such as point clouds and social graphs. H...

--- Documento 2 (Score Re-rank: 6.1202) ---
Título: Neural Bipartite Matching
Categorías: ['cs.LG', 'stat.ML']
Abstract (primeras 200 chars): Graph neural networks (GNNs) have found application for learning in the space
of algorithms. However, the algorithms chosen by existing research (sorting,
Breadth-First search, shortest path finding, ...

--- Documento 3 (Score Re-rank: 5.8683) ---
Título: Training Sensitivity in Graph Isomorphism Network
Categorías: ['cs.LG', 'cs.SI', 'stat.ML']
Abstract (primeras 2

## G. Interfaz Web Conversacional

Para la interfaz utilizamos **Streamlit**, ya que se integra de forma nativa con Streamlit Community Cloud (plataforma de despliegue elegida para este proyecto) y ofrece componentes de chat listos para usar (`st.chat_input`, `st.chat_message`).

La aplicación (`app.py`) reutiliza directamente las funciones `retrieve_documents`, `build_context`, `generate_answer` y `rag_query` definidas en las secciones anteriores del notebook. El flujo de la interfaz es:

1. El usuario escribe una consulta en el campo de chat.
2. Se ejecuta `rag_query()`, que recupera documentos del índice FAISS y genera la respuesta con Gemini.
3. La respuesta se muestra en el chat.
4. Debajo de cada respuesta se muestra un panel expandible con las **evidencias** (documentos y fragmentos usados).
5. Si el corpus no tiene información suficiente, el propio LLM lo indica explícitamente (regla 3 del prompt de la sección E), y esto se refleja igual en el chat.

No se implementa memoria conversacional (no es obligatorio): cada consulta se procesa de forma independiente, aunque el historial visual de la conversación se mantiene en pantalla mediante `st.session_state`.

La API Key de Gemini se gestiona mediante **Streamlit Secrets** (`st.secrets`), nunca hardcodeada en el código fuente.

In [18]:
%%writefile app.py
import os
import streamlit as st
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from google import genai

# --------------------------------------------------------------
# Configuración de página
# --------------------------------------------------------------
st.set_page_config(page_title="RAG arXiv Chat", page_icon="📚", layout="wide")
st.title("📚 Chat RAG sobre arXiv Paper Abstracts")
st.caption("Sistema de Recuperación de Información con embeddings + Gemini")

# --------------------------------------------------------------
# API Key desde Streamlit Secrets (NUNCA hardcodeada)
# En Streamlit Cloud: Settings -> Secrets -> GOOGLE_API_KEY = "tu_key"
# --------------------------------------------------------------
GOOGLE_API_KEY = st.secrets.get("GOOGLE_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    st.error("❌ No se encontró GOOGLE_API_KEY en los Secrets de Streamlit.")
    st.stop()

client = genai.Client(api_key=GOOGLE_API_KEY)
LLM_MODEL = "gemini-2.5-flash"

RAG_PROMPT_TEMPLATE = """Eres un asistente experto en literatura científica. Tu tarea es responder la consulta del usuario ÚNICAMENTE utilizando la información contenida en los documentos de contexto proporcionados a continuación.

Reglas:
1. Si el contexto contiene información suficiente, responde de forma clara y completa, integrando información de varios documentos si es necesario.
2. Cuando uses información de un documento, referencia su número entre corchetes, por ejemplo [Doc 1], [Doc 2].
3. Si el contexto NO contiene información suficiente para responder la consulta, indícalo explícitamente diciendo: "El corpus no contiene información suficiente para responder esta consulta con certeza." No inventes información que no esté en el contexto.

--- CONTEXTO ---
{context}
--- FIN DEL CONTEXTO ---

Consulta del usuario: {query}

Respuesta:"""


# --------------------------------------------------------------
# Carga de embeddings + índice FAISS (una sola vez, cacheado)
# --------------------------------------------------------------
class BGEEmbeddings(Embeddings):
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        instruccion = "Represent this sentence for searching relevant passages: "
        return self.model.encode(instruccion + text, normalize_embeddings=True).tolist()


from huggingface_hub import snapshot_download

@st.cache_resource(show_spinner="Descargando índice FAISS desde Hugging Face...")
def load_vector_store():
    local_path = snapshot_download(
        repo_id="Bryan23y/rag-arxiv-faiss",
        repo_type="dataset",
    )
    embedding_function = BGEEmbeddings()
    vector_store = FAISS.load_local(
        local_path,
        embedding_function,
        allow_dangerous_deserialization=True,
    )
    return vector_store

vector_store = load_vector_store()

# --------------------------------------------------------------
# Funciones del pipeline RAG (idénticas a las del notebook)
# --------------------------------------------------------------
def retrieve_documents(query, k=5):
    docs_with_scores = vector_store.similarity_search_with_score(query, k=k)
    return docs_with_scores  # retorna (doc, score) para poder exponer la evidencia completa


def build_context(retrieved_documents):
    context_parts = []
    for i, doc in enumerate(retrieved_documents):
        titulo = doc.metadata.get("title", "N/A")
        categorias = doc.metadata.get("categories", "N/A")
        abstract = doc.page_content
        context_parts.append(
            f"[Doc {i+1}] Título: {titulo}\nCategorías: {categorias}\nAbstract: {abstract}"
        )
    return "\n\n".join(context_parts)


def generate_answer(query, context):
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, query=query)
    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config={
            "temperature": 0.2,
            "system_instruction": "Eres un asistente de investigación riguroso y honesto.",
        },
    )
    return response.text


def rag_query(query, k=5):
    docs_with_scores = retrieve_documents(query, k=k)
    retrieved_documents = [doc for doc, _ in docs_with_scores]
    context = build_context(retrieved_documents)
    answer = generate_answer(query, context)
    evidencias = [
        {
            "titulo": doc.metadata.get("title", "N/A"),
            "categorias": doc.metadata.get("categories", "N/A"),
            "fragmento": doc.page_content[:300] + "...",
            "score": round(1 - (float(score) ** 2) / 2, 4),  # distancia L2 -> similitud coseno
        }
        for doc, score in docs_with_scores
    ]
    return {"query": query, "respuesta": answer, "evidencias": evidencias}


# --------------------------------------------------------------
# Estado del chat (solo historial visual, sin memoria conversacional real)
# --------------------------------------------------------------
if "messages" not in st.session_state:
    st.session_state.messages = []

# Slider para top-k en la barra lateral
with st.sidebar:
    st.header("⚙️ Configuración")
    k = st.slider("Número de documentos a recuperar (k)", 1, 10, 5)
    st.markdown("---")
    st.markdown(
        "**Sobre este sistema:**\n\n"
        "Corpus: arXiv Paper Abstracts (Kaggle)\n\n"
        "Embeddings: BAAI/bge-small-en-v1.5\n\n"
        "Vector store: FAISS\n\n"
        "LLM: Gemini 2.5 Flash"
    )

# Renderizar historial
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and "evidencias" in msg:
            with st.expander("📄 Ver evidencias utilizadas"):
                for i, ev in enumerate(msg["evidencias"]):
                    st.markdown(f"**[Doc {i+1}] {ev['titulo']}**")
                    st.caption(f"Categorías: {ev['categorias']} | Similitud coseno: {ev['score']}")
                    st.write(ev["fragmento"])
                    st.markdown("---")

# --------------------------------------------------------------
# Input de chat
# --------------------------------------------------------------
if query := st.chat_input("Escribe tu consulta sobre artículos científicos..."):
    st.session_state.messages.append({"role": "user", "content": query})
    with st.chat_message("user"):
        st.markdown(query)

    with st.chat_message("assistant"):
        with st.spinner("Buscando documentos relevantes y generando respuesta..."):
            resultado = rag_query(query, k=k)
        st.markdown(resultado["respuesta"])
        with st.expander("📄 Ver evidencias utilizadas"):
            for i, ev in enumerate(resultado["evidencias"]):
                st.markdown(f"**[Doc {i+1}] {ev['titulo']}**")
                st.caption(f"Categorías: {ev['categorias']} | Similitud coseno: {ev['score']}")
                st.write(ev["fragmento"])
                st.markdown("---")

    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": resultado["respuesta"],
            "evidencias": resultado["evidencias"],
        }
    )


Writing app.py


In [19]:
!ls -la app.py

-rw-r--r-- 1 root root 7468 Jul 16 00:22 app.py


## H. Despliegue en la nube

La aplicación se desplegó utilizando dos servicios complementarios:

1. **Hugging Face Hub** (como *dataset* privado/público) almacena el índice
   vectorial FAISS (embeddings + metadata de los 41,096 documentos). Se eligió
   este servicio en vez de subir el índice directamente al repositorio de
   código, ya que GitHub no está pensado para archivos binarios grandes y
   Streamlit Cloud no persiste archivos entre reinicios del contenedor.
2. **Streamlit Community Cloud** aloja la aplicación (`app.py`), conectada a
   un repositorio de GitHub. En el arranque, la app descarga el índice desde
   Hugging Face Hub mediante `huggingface_hub.snapshot_download` y lo cachea
   con `@st.cache_resource` para no volver a descargarlo en cada consulta.

La API Key de Gemini (`GOOGLE_API_KEY`) se configura como **Streamlit Secret**
desde el panel de administración de la app (Settings → Secrets), nunca
almacenada en el código fuente ni en el repositorio.

A continuación se documenta la subida del índice FAISS a Hugging Face Hub:


In [20]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

login()  # pega aquí tu token cuando te lo pida

In [21]:
api = HfApi()

# Cambia "tu-usuario" por tu nombre de usuario real de Hugging Face
REPO_ID = "Bryan23y/rag-arxiv-faiss"

api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=False, exist_ok=True)

api.upload_folder(
    folder_path="models/faiss_index",
    repo_id=REPO_ID,
    repo_type="dataset",
)

print("✅ Índice subido a Hugging Face:", f"https://huggingface.co/datasets/{REPO_ID}")

✅ Índice subido a Hugging Face: https://huggingface.co/datasets/Bryan23y/rag-arxiv-faiss


### Despliegue final y URL pública

Tras subir el índice a Hugging Face Hub, se subió `app.py` (junto con su
`requirements.txt`) a un repositorio de GitHub, el cual se conectó desde
Streamlit Community Cloud (New app → seleccionar repositorio → rama → archivo
principal `app.py`). Una vez configurado el secreto `GOOGLE_API_KEY` en
Settings → Secrets, la plataforma construyó el entorno automáticamente a
partir de `requirements.txt` y desplegó la aplicación.

**URL pública del sistema desplegado:**
[https://examen-final-debvy2oa93cdyd2mq57rym.streamlit.app](https://examen-final-debvy2oa93cdyd2mq57rym.streamlit.app)

El sistema permanece activo y accesible desde cualquier navegador, sin
necesidad de instalar dependencias, y admite cualquier consulta en lenguaje
natural relacionada con el corpus de arXiv Paper Abstracts.

**Contenido de `requirements.txt` utilizado para el despliegue:**

```
streamlit
sentence-transformers
langchain
langchain-community
langchain-core
faiss-cpu
huggingface_hub
google-genai
```


## I. Evaluación del sistema

Para evaluar el sistema de forma cualitativa, se seleccionaron 4 consultas de prueba que cubren distintos escenarios:

1. Una consulta directa y bien representada en el corpus (GNNs).
2. Una consulta que requiere integrar información de varios documentos (RL en robótica).
3. Una consulta sobre un tema de vanguardia dentro del dominio (modelos de difusión).
4. Una consulta **fuera del dominio del corpus**, para verificar que el sistema reconoce correctamente la falta de información.

Para cada consulta se documenta un juicio subjetivo sobre 5 criterios:

- **Corrección**: ¿la respuesta es factualmente correcta según los abstracts recuperados?
- **Relevancia**: ¿la respuesta atiende directamente lo que pidió la consulta?
- **Fidelidad**: ¿todo lo afirmado está respaldado por las evidencias mostradas, sin alucinaciones?
- **Integración**: ¿el sistema combina información de más de un documento cuando corresponde?
- **Reconocimiento de información insuficiente**: ¿el sistema indica explícitamente cuando el corpus no puede responder la consulta, en vez de inventar una respuesta?

In [33]:
import time
from google.genai.errors import ClientError

# --------------------------------------------------------------
# Evaluación cualitativa del sistema RAG (Con Reintento Automático Blindado)
# --------------------------------------------------------------

consultas_evaluacion = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.", # <-- AGREGADA
    "What is the traditional recipe for Ecuadorian ceviche?",  # Fuera de dominio (anti-alucinación)
]

resultados_evaluacion = []

print("🚀 Iniciando evaluación con sistema de protección de cuota (Rate Limit)...")

for i, consulta in enumerate(consultas_evaluacion):
    print("=" * 80)
    print(f"▶️ [{i+1}/{len(consultas_evaluacion)}] Procesando consulta...")

    exito = False
    intento = 1

    # Bucle de reintento: Si la API da error 429, espera y vuelve a intentar
    while not exito:
        try:
            resultado = rag_query(consulta, k=5)
            exito = True  # Si funcionó, salimos del bucle while
        except ClientError as e:
            # Blindaje: verificamos el atributo .code o si el texto menciona el límite 429 / RESOURCE_EXHAUSTED
            error_msg = str(e)
            error_code = getattr(e, 'code', None)

            if error_code == 429 or "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                print(f"⚠️ Cuota de API alcanzada (Intento {intento}). Pausando 45 segundos para que los servidores se liberen...")
                time.sleep(45)
                intento += 1
            else:
                # Si es un error real (ej. problemas de conexión o API Key rota), lo mostramos
                raise e

    resultados_evaluacion.append(resultado)
    print(f"\nCONSULTA: {consulta}")
    print(f"\nRESPUESTA:\n{resultado['respuesta']}")
    print(f"\nDocumentos recuperados: {len(resultado['evidencias'])}")
    print()

    # Pausa ligera entre consultas exitosas para mantener el flujo estable
    if i < len(consultas_evaluacion) - 1:
        time.sleep(10)

print("=" * 80)
print("✅ ¡Evaluación completada con éxito!")

🚀 Iniciando evaluación con sistema de protección de cuota (Rate Limit)...
▶️ [1/5] Procesando consulta...

🔍 Recuperados y re-ordenados 5 documentos para: 'What are the main applications of Graph Neural Networks?'

--- Documento 1 (Score Re-rank: 7.5404) ---
Título: Hierarchical Bipartite Graph Convolution Networks
Categorías: ['cs.LG', 'cs.CV', 'stat.ML']
Abstract (primeras 200 chars): Recently, graph neural networks have been adopted in a wide variety of
applications ranging from relational representations to modeling irregular data
domains such as point clouds and social graphs. H...

--- Documento 2 (Score Re-rank: 6.1202) ---
Título: Neural Bipartite Matching
Categorías: ['cs.LG', 'stat.ML']
Abstract (primeras 200 chars): Graph neural networks (GNNs) have found application for learning in the space
of algorithms. However, the algorithms chosen by existing research (sorting,
Breadth-First search, shortest path finding, ...

--- Documento 3 (Score Re-rank: 5.8683) ---
Título: Traini

## I. Evaluación del sistema

### Resultados de la evaluación

| # | Consulta | Corrección | Relevancia | Fidelidad | Integración | Reconoce info. insuficiente | Comentarios |
|---|---|:---:|:---:|:---:|:---:|:---:|---|
| **1** | **Applications of GNNs** | Alta | Alta | Alta | Sí<br>*(Doc 1,2,3,4,5)* | N/A | El re-ranker colocó en el Top-1 un documento general sobre adopción de GNNs en dominios irregulares (score 7.54). La respuesta integra los 5 documentos para cubrir clasificación, grafos bipartitos y redes inalámbricas. |
| **2** | **RL en robótica** | Alta | Alta | Alta | Sí<br>*(Doc 1,2,3,4,5)* | N/A | Es la respuesta más completa de las cinco: estructura la información en categorías claras (aplicaciones, control end-to-end, eficiencia de datos y alternativas a recompensas). Cada afirmación está anclada con precisión a su respectivo `[Doc N]`. |
| **3** | **Modelos de difusión** | Alta | Alta | Alta | Sí<br>*(Doc 1,2,3,4,5)* | N/A | Respuesta técnica y detallada. El re-ranker separó un paper fundamental (score 8.38) de otros más específicos (scores ~3.5). Explica métodos como UDM, DDIM y TNRD sin mezclar conceptos entre documentos. |
| **4** | **Mejoras en sistemas RAG** | Alta | Media-Alta | Alta | Sí<br>*(Doc 2,3)* | N/A | Aunque el corpus no tiene papers de "RAG moderno con LLMs" (el concepto era incipiente en este dataset de arXiv), el re-ranker priorizó exitosamente arquitecturas híbridas de recuperación-generación en visión por computador y NLP (*HRGR-Agent*). |
| **5** | **Receta de ceviche ecuatoriano** | N/A | Alta | Alta | N/A | **Sí** | El sistema superó la prueba anti-alucinación. El re-ranker asignó **puntajes fuertemente negativos (-10.9 a -11.1)** a los documentos de comida recuperados por FAISS, y el LLM respetó la Regla 3 al declarar explícitamente la falta de información. |

---

### Conclusiones y Análisis Técnico

* **Fidelidad y Trazabilidad:** El sistema demuestra una fidelidad del 100% respecto al contexto. En las 4 consultas del dominio, no se detectaron alucinaciones ni extrapolaciones externas; el modelo sustentó cada punto técnico citando rigurosamente la evidencia mediante etiquetas `[Doc N]`.
* **Capacidad de Integración y Síntesis:** El LLM no se limita a resumir los abstracts de forma aislada. En consultas amplias como la de robótica (Consulta 2) y modelos de difusión (Consulta 3), el sistema organizó la información temáticamente (ej. *Control End-to-End*, *Eficiencia de Datos*, *Recompensas Dispersas*), fusionando aportes de hasta 5 artículos científicos en una respuesta cohesiva.
* **Impacto Crítico del Re-ranking (Cross-Encoder):** La adición del modelo `ms-marco-MiniLM-L-6-v2` como segunda etapa de recuperación superó las limitaciones de la similitud semántica superficial de los embeddings:
    * **En la consulta fuera de dominio (Consulta 5):** Mientras que la búsqueda vectorial inicial (FAISS) devolvió artículos "parecidos" por palabras clave (recetas, pizzas, análisis de comida), el Cross-Encoder identificó que no respondían a la pregunta y los penalizó con **scores negativos extremos (-10.96 a -11.18)**. Esto demuestra que el re-ranker es un excelente filtro de ruido.
    * **Jerarquización de relevancia:** En la Consulta 3, el re-ranker identificó correctamente el documento más general y relevante sobre avances en SDEs para difusión, otorgándole un score muy alto (**8.38**), mientras que los métodos de aplicación hiper-específica recibieron scores moderados (**2.7 a 3.5**).
* **Comportamiento ante conceptos modernos (Consulta 4):** Al evaluar técnicas para sistemas RAG, el sistema evidenció un límite temporal del corpus (el dataset no contiene literatura masiva sobre RAG con LLMs post-2023). Sin embargo, el re-ranker logró rescatar exitosamente literatura paralela sobre *Sistemas Híbridos de Recuperación-Generación* en reportes médicos y NLP, ofreciendo una respuesta conceptualmente válida y honesta con la literatura disponible.
* **Mejora arquitectónica propuesta:** Al observar que el Cross-Encoder asigna puntajes negativos a documentos irrelevantes (como se vio en la Consulta 5), una mejora futura para el pipeline sería **establecer un umbral de corte (threshold en el Re-ranker)**. Por ejemplo, si el score máximo de una consulta es menor a `0.0`, el sistema podría descartar el contexto automáticamente y emitir el mensaje de "información insuficiente" sin necesidad de gastar tokens ni invocar a la API del LLM.